In [215]:
import pandas as pd
import numpy as np
import re

In [216]:
# Load the raw dataset
df = pd.read_csv("../data/jobstreet_all_job_dataset.csv")
print("Dataset loaded successfully")

Dataset loaded successfully


In [217]:
# Create a separate copy for cleaning
# All preprocessing steps will be applied to df_clean
df_clean = df.copy()

In [218]:
# Check the data types of all columns in the cleaned working copy
df_clean.dtypes

job_id          float64
job_title           str
company             str
descriptions        str
location            str
category            str
subcategory         str
role                str
type                str
salary              str
listingDate         str
dtype: object

In [219]:
# Convert listingDate into datetime format
df_clean["listingDate"] = pd.to_datetime(df_clean["listingDate"], errors="coerce")
print(df_clean["listingDate"].isna().sum())

0


In [220]:
# Convert job_id into integer
df_clean["job_id"] = df_clean["job_id"].astype("Int64").astype(str)
df_clean["job_id"].head()

0    74630583
1    74660602
2    74655679
3    74657624
4    74679363
Name: job_id, dtype: str

In [221]:
# Recheck duplicate records after data type conversion
print("Duplicate job_id:", df_clean["job_id"].duplicated().sum())
print("Full duplicated rows:", df_clean.duplicated().sum())

Duplicate job_id: 0
Full duplicated rows: 0


In [222]:
# Remove leading and trailing spaces from text columns
text_cols = [
    "job_title", "company", "descriptions", "location", "category",
    "subcategory", "role", "type", "salary"
]

for col in text_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()
    df_clean[col] = df_clean[col].replace("nan", np.nan)

df_clean[text_cols].head()

,job_title,company,descriptions,location,category,subcategory,role,type,salary
0,Procurement Executive (Contract),Coca-Cola Bottlers (Malaysia) Sdn Bhd,"Position Purpose\nManage aspects of procurement including supplier management, contract management, and vendor performance tracking. Support the administration and management of procurement processes. This includes setting up, maintaining, and administering purchasing databases; developing and issuing purchase orders; maintain contract log and providing support to other departments within the organization who are involved in procurement.\nKey Duties / Responsibilities\nGeneral\nEnsure the PO created following the bidding requirements\nOA updates from time to time - upon request from planning team / buyers\nGeographical coverage: Malaysia and Singapore operations\nEnsure all purchasing activities comply with Procurement Procedure and company internal policy\nActively seek out opportunities for improvement, ways to do things better; and lead implementation of actions defined\nManage reports and reviews\nManage reports and reviews weekly, monthly and quarterly accordingly\nContract Log Update; Ensure expiring / expired contracts are renewed on time; Safekeeping of contracts softcopy\nActive and Inactive Vendor Code Cleansing on quarterly basis\nSupplier management:\nDefine and implement strategy for supplier management to ensure reliability in consistently meeting requirements, quality control and product performance per contract, continuous improvement in cost, quality and service.\nVendor Creation follows the policy and acknowledgment from respective buyers\nDocumentation and Audit:\nKeep record of PAF, PAF approvals, Bidding documents, Contracts softcopies\nWork closely with Legal in contract storage and contract log\nKeep track on all report as evidence for audit.\nRequirements / Qualifications\nEducation:\nBachelor Degree preferably in procurement and/or supply chain related field. Professional knowledge of procurement practices, supply markets, terms, negotiation concepts and business laws.\nExperience:\n1-2 years’ experience relevant experience in FMCG/beverage manufacturing environment or any other related field with a strong procurement background in Malaysia.\nLanguage\n: Bahasa Melayu/ English\nWork Location\n: Bandar Enstek (Negeri Sembilan)\nEmployment\n: 12 months Contract\nWith Regret , only shortlisted candidate will be notified.",Negeri Sembilan,"Manufacturing, Transport & Logistics","Purchasing, Procurement & Inventory",procurement-executive,Contract/Temp,NaN
1,Account Executive/ Assistant,Acoustic & Lighting System Sdn Bhd,"We are looking for a Account Executive/ Assistant to join our team.\nWhat you will get when you join us:\nFair work life balance – Mon to Friday, 9am - 6pm\nInsurance & medical coverage\nSport facilities\nResponsibilities:\nResponsible for daily accounts recording functions.\nAssist in overall filing of accounts related documents and maintenance of records.\nTo ensure the accounts payable/receivable, bank reconciliation, journal and general ledger functions are performed in an effective and efficient manner.\nResponsible for the timely submission, accuracy and maintenance of financial records.\nMonitor payment and installments.\nIssuing DN/CN, payment vouchers and journal voucher.\nThe scope of work should be revised from time to time based on management needs or requirements.\nJob Requirements:\nAt least Diploma/ Degree in Finance/Accountancy/Banking or equivalent.\nFresh Graduate are encouraged to apply.\nPrefer candidates that can speak fluent in\nMandarin\nis an added advantage.",Petaling,Accounting,Bookkeeping & Small Practice Accounting,executive-assistant,Full time,"RM 2,800 – RM 3,200 per month"
2,"Data Analyst - Asset Management, SPX Express",Shopee Mobile Malaysia Sdn Bhd,"Performs detailed data analysis on existing spendings / consumptions of SPX and identifies cost saving areas.\nPlan out and rollout cost saving initiatives to re

In [223]:
# Clean and standardize values in the type column
def clean_type_value(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    
    # remove square brackets
    x = x.replace("[", "").replace("]", "")
    
    # remove quotes
    x = x.replace("'", "").replace('"', "")
    
    # normalize spaces around commas
    x = re.sub(r"\s*,\s*", ", ", x)
    
    return x.strip()

df_clean["type_clean"] = df_clean["type"].apply(clean_type_value)

df_clean[["type", "type_clean"]].sample(10, random_state=42)

,type,type_clean
14886,Full time,Full time
21842,Contract/Temp,Contract/Temp
53575,Full time,Full time
50853,Full time,Full time
5662,Full time,Full time
57533,Full time,Full time
56381,Full time,Full time
36229,Full time,Full time
20851,Full time,Full time
836,Full time,Full time


In [224]:
# Review the frequency distribution of the cleaned employment type values
df_clean["type_clean"].value_counts(dropna=False).head(20)

type_clean
Full time                              62820
Contract/Temp                           5560
Part time                                367
Casual/Vacation                          273
Contract/Temp, Full time                   2
Full time, Part time                       1
Contract/Temp, Full time, Part time        1
Name: count, dtype: int64

In [225]:
# Replace missing values in the role column with a placeholder label
df_clean["role_clean"] = df_clean["role"].fillna("Unknown").str.strip()

# Review the most frequent values in the cleaned role column
df_clean["role_clean"].value_counts().head(10)

role_clean
Unknown                     2252
account-executive           1483
sales-executive             1221
accounts-executive          1198
administration-officer      1116
Executive                    951
human-resource-executive     919
administrative-executive     826
finance-executive            736
marketing-executive          643
Name: count, dtype: int64

In [226]:
# Standardize basic formatting in the location column
df_clean["location_clean"] = df_clean["location"].str.strip()

# remove repeated spaces
df_clean["location_clean"] = df_clean["location_clean"].str.replace(r"\s+", " ", regex=True)

# standardize comma spacing
df_clean["location_clean"] = df_clean["location_clean"].str.replace(r"\s*,\s*", ", ", regex=True)

df_clean[["location", "location_clean"]].sample(10, random_state=42)

,location,location_clean
14886,Seberang Perai,Seberang Perai
21842,Kuala Lumpur,Kuala Lumpur
53575,Mont Kiara,Mont Kiara
50853,Kuala Lumpur,Kuala Lumpur
5662,Kuala Lumpur,Kuala Lumpur
57533,Selangor,Selangor
56381,Sepang,Sepang
36229,Penang Island,Penang Island
20851,Klang District,Klang District
836,Kuala Lumpur,Kuala Lumpur


In [227]:
# Review the most frequent values in the cleaned location column
df_clean["location_clean"].value_counts().head(30)

location_clean
Kuala Lumpur                              15390
Petaling                                   9746
Johor Bahru District                       4480
Selangor                                   2842
Shah Alam/Subang                           2621
Kuala Lumpur City Centre                   2112
Penang                                     1930
Penang Island                              1885
Seberang Perai                             1796
Klang District                             1427
Hulu Langat                                 958
Kajang/Bangi/Serdang                        937
Sepang                                      865
Kota Kinabalu District                      805
Kinta District                              776
Kuching Division                            770
Johor                                       729
Seremban District                           638
Kulai District                              628
Petaling Jaya, Selangor                     519
Bangsar South            

In [228]:
# Standardize salary formatting
df_clean["salary_clean_text"] = df_clean["salary"].copy()

# remove extra spaces
df_clean["salary_clean_text"] = df_clean["salary_clean_text"].str.strip()
df_clean["salary_clean_text"] = df_clean["salary_clean_text"].str.replace(r"\s+", " ", regex=True)

# standardize dash symbols
df_clean["salary_clean_text"] = df_clean["salary_clean_text"].str.replace("–", "-", regex=False)
df_clean["salary_clean_text"] = df_clean["salary_clean_text"].str.replace("—", "-", regex=False)

# standardize currency text
df_clean["salary_clean_text"] = df_clean["salary_clean_text"].str.replace("MYR", "RM", regex=False)

# standardize monthly / yearly wording
df_clean["salary_clean_text"] = df_clean["salary_clean_text"].str.replace("p.m.", "per month", regex=False)
df_clean["salary_clean_text"] = df_clean["salary_clean_text"].str.replace("p.a.", "per year", regex=False)

df_clean[["salary", "salary_clean_text"]].sample(15, random_state=42)

,salary,salary_clean_text
14886,NaN,NaN
21842,NaN,NaN
53575,"RM 2,300 – RM 2,500 per month","RM 2,300 - RM 2,500 per month"
50853,NaN,NaN
5662,NaN,NaN
57533,NaN,NaN
56381,NaN,NaN
36229,"RM 2,200 – RM 3,000 per month","RM 2,200 - RM 3,000 per month"
20851,NaN,NaN
836,NaN,NaN


In [229]:
# Identify the salary period from the cleaned salary text
def detect_salary_period(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).lower()
    
    if "per month" in x or "/month" in x or "monthly" in x:
        return "monthly"
    elif "per year" in x or "/year" in x or "annual" in x or "annum" in x:
        return "yearly"
    elif "per hour" in x or "/hour" in x or "hourly" in x:
        return "hourly"
    else:
        return "unknown"

# Create a new column for salary period classification
df_clean["salary_period"] = df_clean["salary_clean_text"].apply(detect_salary_period)

df_clean["salary_period"].value_counts(dropna=False)

salary_period
NaN        37430
monthly    30468
unknown      924
yearly       152
hourly        50
Name: count, dtype: int64

In [230]:
# Extract minimum and maximum salary values from the cleaned salary text
def extract_salary_range(x):
    if pd.isna(x):
        return pd.Series([np.nan, np.nan])
    
    x = str(x).lower()
    
    # handle k notation like 9k or 10.5k
    x = re.sub(r'(\d+(?:\.\d+)?)\s*k', lambda m: str(float(m.group(1)) * 1000), x)
    
    # extract numbers from the salary text
    nums = re.findall(r'\d[\d,]*\.?\d*', x)
    nums = [float(n.replace(",", "")) for n in nums]
    
    if len(nums) == 0:
        return pd.Series([np.nan, np.nan])
    elif len(nums) == 1:
        return pd.Series([nums[0], nums[0]])
    else:
        return pd.Series([nums[0], nums[1]])

# Create separate columns for minimum and maximum salary values
df_clean[["salary_min", "salary_max"]] = df_clean["salary_clean_text"].apply(extract_salary_range)

df_clean[["salary_clean_text", "salary_min", "salary_max"]].sample(15, random_state=42)

,salary_clean_text,salary_min,salary_max
14886,NaN,NaN,NaN
21842,NaN,NaN,NaN
53575,"RM 2,300 - RM 2,500 per month",2300.0,2500.0
50853,NaN,NaN,NaN
5662,NaN,NaN,NaN
57533,NaN,NaN,NaN
56381,NaN,NaN,NaN
36229,"RM 2,200 - RM 3,000 per month",2200.0,3000.0
20851,NaN,NaN,NaN
836,NaN,NaN,NaN


In [231]:
# Review the statistical summary of the extracted salary range values
df_clean[["salary_min", "salary_max"]].describe()

,salary_min,salary_max
count,31505.000000,31505.000000
mean,4225.183158,5722.901460
std,7593.412290,9804.718204
min,1.000000,1.000000
25%,2500.000000,3500.000000
50%,3200.000000,4500.000000
75%,4500.000000,6000.000000
max,450000.000000,500000.000000


In [232]:
# Calculate the salary midpoint from the extracted minimum and maximum values
df_clean["salary_mid"] = (df_clean["salary_min"] + df_clean["salary_max"]) / 2
df_clean[["salary_clean_text", "salary_min", "salary_max", "salary_mid"]].sample(15, random_state=42)

,salary_clean_text,salary_min,salary_max,salary_mid
14886,NaN,NaN,NaN,NaN
21842,NaN,NaN,NaN,NaN
53575,"RM 2,300 - RM 2,500 per month",2300.0,2500.0,2400.0
50853,NaN,NaN,NaN,NaN
5662,NaN,NaN,NaN,NaN
57533,NaN,NaN,NaN,NaN
56381,NaN,NaN,NaN,NaN
36229,"RM 2,200 - RM 3,000 per month",2200.0,3000.0,2600.0
20851,NaN,NaN,NaN,NaN
836,NaN,NaN,NaN,NaN


In [233]:
# Review the statistical summary of the salary midpoint
df_clean["salary_mid"].describe()

count     31505.000000
mean       4974.042309
std        8645.786981
min           1.000000
25%        3000.000000
50%        3750.000000
75%        5000.000000
max      475000.000000
Name: salary_mid, dtype: float64

In [234]:
# Convert salary midpoint values into a monthly salary estimate
def to_monthly_salary(mid_salary, period):
    if pd.isna(mid_salary) or pd.isna(period):
        return np.nan
    
    if period == "monthly":
        return mid_salary
    elif period == "yearly":
        return mid_salary / 12
    elif period == "hourly":
        return mid_salary * 8 * 22
    else:
        return np.nan

# Create a monthly normalized salary variable
df_clean["salary_monthly_est"] = df_clean.apply(
    lambda row: to_monthly_salary(row["salary_mid"], row["salary_period"]),
    axis=1
)

df_clean[["salary_clean_text", "salary_period", "salary_mid", "salary_monthly_est"]].sample(20, random_state=42)

,salary_clean_text,salary_period,salary_mid,salary_monthly_est
14886,NaN,NaN,NaN,NaN
21842,NaN,NaN,NaN,NaN
53575,"RM 2,300 - RM 2,500 per month",monthly,2400.0,2400.0
50853,NaN,NaN,NaN,NaN
5662,NaN,NaN,NaN,NaN
57533,NaN,NaN,NaN,NaN
56381,NaN,NaN,NaN,NaN
36229,"RM 2,200 - RM 3,000 per month",monthly,2600.0,2600.0
20851,NaN,NaN,NaN,NaN
836,NaN,NaN,NaN,NaN


In [235]:
# Review the statistical summary of the monthly normalized salary
df_clean["salary_monthly_est"].describe()

count     30669.000000
mean       4701.304880
std        6886.839917
min           1.000000
25%        3000.000000
50%        3750.000000
75%        5000.000000
max      440000.000000
Name: salary_monthly_est, dtype: float64

In [236]:
df_clean["salary_period"].value_counts(dropna=False)

salary_period
NaN        37430
monthly    30468
unknown      924
yearly       152
hourly        50
Name: count, dtype: int64

In [237]:
# Inspect salary records classified as unknown to understand their patterns
unknown_salary_sample = df_clean.loc[
    df_clean["salary_period"] == "unknown",
    ["job_title", "salary_clean_text", "salary_min", "salary_max", "salary_mid"]
]

unknown_salary_sample.sample(30, random_state=42)

,job_title,salary_clean_text,salary_min,salary_max,salary_mid
19503,Account Assistant,"RM 2,300 - 3,000",2300.0,3000.0,2650.0
61125,Insurance Specialist (Banking - Malaysia Market),RM5000 - RM6500,5000.0,6500.0,5750.0
1447,Talent Acquisition Specialist,AWS,NaN,NaN,NaN
58562,Sales Operations Executive,"RM 3,000 - 4,500",3000.0,4500.0,3750.0
18305,Engineer/ Senior Engineer_ Research & Development (Bukit Beruntung),"RM 3,000 - 4,200",3000.0,4200.0,3600.0
3562,Admin Assistant,"RM 2,000 - 3,000",2000.0,3000.0,2500.0
10153,Finance Executive,"RM 3,500 - 4,500",3500.0,4500.0,4000.0
8170,Finance Officer,"RM 3,000 - 3,300",3000.0,3300.0,3150.0
39428,OPERATION BACK OFFICE,"RM 2,700 - 4,000",2700.0,4000.0,3350.0
62336,Human Resource Executive - Talent Acquisition,"RM 3,500 - 4,000",3500.0,4000.0,3750.0


In [238]:
# Review raw salary text for records classified as unknown
df_clean.loc[
    df_clean["salary_period"] == "unknown",
    "salary_clean_text"
].head(50)

32                     RM 6,500 - 7,500
93                     RM 4,500 - 6,000
94                     RM 2,000 - 2,500
134                    RM 2,000 - 3,000
375                    RM 3,000 - 3,500
386                    RM 2,400 - 3,400
413                    RM 2,700 - 3,800
418                    RM 2,400 - 3,360
431                    RM 2,500 - 3,500
448                    RM 2,500 - 3,500
478                      RM 800 - 1,000
515                    RM 2,500 - 3,500
595                    RM 3,000 - 4,000
627                    RM 2,600 - 3,000
784                    RM 2,000 - 2,800
833                    RM 3,000 - 4,500
899                    RM 3,500 - 4,900
940                    RM 2,500 - 3,500
973                    RM 2,500 - 3,500
981                    RM 5,500 - 8,000
1068                   RM 3,000 - 3,500
1088                              6,000
1177                       RM8k - RM10k
1191    Attractive Remuneration Package
1193                   RM 3,000 - 4,200


In [239]:
# Flag records where a numeric salary value was successfully extracted
df_clean["salary_has_number"] = df_clean["salary_min"].notna().astype(int)
df_clean["salary_has_number"].value_counts()

salary_has_number
0    37519
1    31505
Name: count, dtype: int64

In [240]:
# Inspect non missing salary entries that do not contain extractable numeric values
df_clean.loc[
    (df_clean["salary"].notna()) & (df_clean["salary_has_number"] == 0),
    "salary_clean_text"
].value_counts().head(30)

salary_clean_text
bonuses                                     14
Transport,Dental,Optical,Meal Allowance      9
Competitive Remuneration & Benefits          8
Competitive                                  5
Attractive remuneration package              3
Healthcare, Vacation subsidy                 3
Healthcare, Travel Subsidy                   3
Competitive Salary and Benefits              2
hybrid                                       2
Competitive Salary Package                   2
Competitive Salary                           2
Attractive                                   2
Confidential                                 2
-                                            2
Competitive salary and benefits.             2
Attractive Remuneration Package              1
AWS                                          1
Salaried Monthly                             1
Attractive Salary Package                    1
Extended employee benefits and perks         1
Competitive salary and benefits package   

In [241]:
# Reclasify unknown salary period that has value into monthly
mask_unknown_numeric = (
    (df_clean["salary_period"] == "unknown") &
    (df_clean["salary_has_number"] == 1)
)

df_clean.loc[mask_unknown_numeric, "salary_period"] = "monthly"

In [242]:
# Recalculate monthly salary estimates after salary period refinement
df_clean["salary_monthly_est"] = df_clean.apply(
    lambda row: to_monthly_salary(row["salary_mid"], row["salary_period"]),
    axis=1
)

In [243]:
# Review the updated salary period distribution
df_clean["salary_period"].value_counts(dropna=False)

salary_period
NaN        37430
monthly    31304
yearly       152
unknown       88
hourly        50
Name: count, dtype: int64

In [244]:
# Review the updated summary statistics of monthly salary estimates
df_clean["salary_monthly_est"].describe()

count     31505.000000
mean       4703.375888
std        6845.461124
min           1.000000
25%        3000.000000
50%        3750.000000
75%        5000.000000
max      440000.000000
Name: salary_monthly_est, dtype: float64

In [245]:
# Inspect records with the highest monthly estimated salaries
df_clean.loc[
    df_clean["salary_monthly_est"].notna(),
    ["job_title", "salary_clean_text", "salary_period", "salary_min", "salary_max", "salary_mid", "salary_monthly_est"]
].sort_values("salary_monthly_est", ascending=False).head(30)

,job_title,salary_clean_text,salary_period,salary_min,salary_max,salary_mid,salary_monthly_est
55558,Part Time Driver,"RM 2,000 - RM 3,000 per hour",hourly,2000.0,3000.0,2500.0,440000.0
55849,Part Time Plumber,"RM 2,000 - RM 3,000 per hour",hourly,2000.0,3000.0,2500.0,440000.0
57017,Part Time Pet Groomer,"RM 2,000 - RM 3,000 per hour",hourly,2000.0,3000.0,2500.0,440000.0
67019,Freelance Trainers Needed for Online Webinars,"RM 2,000 - RM 3,000 per hour",hourly,2000.0,3000.0,2500.0,440000.0
57111,Part Time Aircond Technician,"RM 2,000 - RM 3,000 per hour",hourly,2000.0,3000.0,2500.0,440000.0
57776,Intern - Mobility Team (Cyberjaya),"RM 1,500 - RM 1,501 per hour",hourly,1500.0,1501.0,1500.5,264088.0
40635,Process Integration Engineer,"RM 200,000 - RM 300,000 per month",monthly,200000.0,300000.0,250000.0,250000.0
31945,Internship - Finance,"RM 800 - RM 1,000 per hour",hourly,800.0,1000.0,900.0,158400.0
9959,Corporate Banking Lending Specialist,"RM 130,000 - RM 150,000 per month",monthly,130000.0,150000.0,140000.0,140000.0
18934,Devops Engineer,$100000.00 - $150000.00 per month,monthly,100000.0,150000.0,125000.0,125000.0


In [246]:
# Inspect records with the lowest monthly estimated salaries
df_clean.loc[
    df_clean["salary_monthly_est"].notna(),
    ["job_title", "salary_clean_text", "salary_period", "salary_min", "salary_max", "salary_mid", "salary_monthly_est"]
].sort_values("salary_monthly_est", ascending=True).head(30)

,job_title,salary_clean_text,salary_period,salary_min,salary_max,salary_mid,salary_monthly_est
57786,Planning Executive,RM 1 per month,monthly,1.0,1.0,1.00,1.000000
58663,Production Executive,RM 1 per month,monthly,1.0,1.0,1.00,1.000000
54723,Executive of Customer Care & Experience,RM 1 - RM 2 per month,monthly,1.0,2.0,1.50,1.500000
52188,Internship for Media/Art Students,RM 1 - 2,monthly,1.0,2.0,1.50,1.500000
33276,Internship for Media/Art Students,RM 1 - 2,monthly,1.0,2.0,1.50,1.500000
48413,HR Operations (Payroll and Inventory),RM 2.000 - RM 3.000 per month,monthly,2.0,3.0,2.50,2.500000
54868,Account Assistant 会计助理,RM 2 - RM 3 per month,monthly,2.0,3.0,2.50,2.500000
62928,E-commerce Sales Executive (High Commission Package),RM 2.800 - RM 3.500 per month,monthly,2.8,3.5,3.15,3.150000
27449,Junior Business Development Manager,RM 2.800 - RM 3.500 per month,monthly,2.8,3.5,3.15,3.150000
67053,Executive - Sales Order Management,RM 3.000 - RM 4.000 per month,monthly,3.0,4.0,3.50,3.500000


In [247]:
# Create a flag column for salary parsing issues
df_clean["salary_parse_issue_flag"] = 0

In [248]:
# Flag salary values that likely use dots as thousand separators
mask_dot_thousand_issue = (
    df_clean["salary_clean_text"].notna() &
    df_clean["salary_clean_text"].str.contains(r"RM\s*\d{1,3}\.\d{3}", regex=True, na=False) &
    df_clean["salary_mid"].notna() &
    (df_clean["salary_mid"] < 100)
)

df_clean.loc[mask_dot_thousand_issue, "salary_parse_issue_flag"] = 1

In [249]:
# Flag suspicious salary entries where numbers may come from non salary text
mask_non_salary_number = (
    df_clean["salary_clean_text"].notna() &
    df_clean["salary_clean_text"].str.contains(r"months|month", case=False, na=False) &
    ~df_clean["salary_clean_text"].str.contains(r"RM|\$|MYR|SGD|฿", case=False, na=False) &
    df_clean["salary_mid"].notna()
)

df_clean.loc[mask_non_salary_number, "salary_parse_issue_flag"] = 1

In [250]:
# Preview records flagged as salary parsing issues
df_clean.loc[
    df_clean["salary_parse_issue_flag"] == 1,
    ["job_title", "salary_clean_text", "salary_period", "salary_min", "salary_max", "salary_mid"]
].head(50)

,job_title,salary_clean_text,salary_period,salary_min,salary_max,salary_mid
63168,Purchasing - Team Lead,x 13 months,monthly,13.0,13.0,13.0
68887,Charging Developer,"Monthly salary 5,000 - 11,000",monthly,5000.0,11000.0,8000.0


In [251]:
# Review the number of flagged records
df_clean["salary_parse_issue_flag"].value_counts()

salary_parse_issue_flag
0    69022
1        2
Name: count, dtype: int64

In [252]:
# Improved salary extraction function to handle dot thousand separators
def extract_salary_range_v2(x):
    if pd.isna(x):
        return pd.Series([np.nan, np.nan])
    
    x = str(x).lower().strip()
    
    # handle k notation first, e.g. 9k -> 9000
    x = re.sub(r'(\d+(?:\.\d+)?)\s*k', lambda m: str(float(m.group(1)) * 1000), x)
    
    # if pattern like 2.800 or 15.000 appears, treat dot as thousands separator
    x = re.sub(r'(?<=\d)\.(?=\d{3}\b)', '', x)
    
    nums = re.findall(r'\d[\d,]*\.?\d*', x)
    nums = [float(n.replace(",", "")) for n in nums]
    
    if len(nums) == 0:
        return pd.Series([np.nan, np.nan])
    elif len(nums) == 1:
        return pd.Series([nums[0], nums[0]])
    else:
        return pd.Series([nums[0], nums[1]])

In [253]:
# Identify salary records that use dots as thousand separators
mask_dot_thousand_issue = (
    df_clean["salary_clean_text"].notna() &
    df_clean["salary_clean_text"].str.contains(r"RM\s*\d{1,3}\.\d{3}", regex=True, na=False)
)
print("Records with dot thousand separators:", mask_dot_thousand_issue.sum())

# Re-extract salary values only if such records exist; apply() on an empty
if mask_dot_thousand_issue.any():
    df_clean.loc[mask_dot_thousand_issue, ["salary_min", "salary_max"]] = (
        df_clean.loc[mask_dot_thousand_issue, "salary_clean_text"]
        .apply(extract_salary_range_v2)
        .values
    )

Records with dot thousand separators: 0


In [254]:
# Preview corrected salary values
df_clean["salary_mid"] = (df_clean["salary_min"] + df_clean["salary_max"]) / 2

df_clean["salary_monthly_est"] = df_clean.apply(
    lambda row: to_monthly_salary(row["salary_mid"], row["salary_period"]),
    axis=1
)

In [255]:
# Preview corrected salary values
df_clean.loc[
    mask_dot_thousand_issue,
    ["job_title", "salary_clean_text", "salary_min", "salary_max", "salary_mid", "salary_monthly_est"]
].head(30)

,job_title,salary_clean_text,salary_min,salary_max,salary_mid,salary_monthly_est


In [256]:
# Identify salary entries that contain compensation related text but not actual salary amounts
mask_invalid_salary_text = (
    df_clean["salary_clean_text"].notna() &
    (
        df_clean["salary_clean_text"].str.contains(r"\b\d+\s*months?\b", case=False, na=False) |
        df_clean["salary_clean_text"].str.contains(r"bonus", case=False, na=False) |
        df_clean["salary_clean_text"].str.contains(r"allowance", case=False, na=False) |
        df_clean["salary_clean_text"].str.contains(r"remuneration", case=False, na=False) |
        df_clean["salary_clean_text"].str.contains(r"benefit", case=False, na=False) |
        df_clean["salary_clean_text"].str.contains(r"competitive", case=False, na=False) |
        df_clean["salary_clean_text"].str.contains(r"attractive", case=False, na=False) |
        df_clean["salary_clean_text"].str.contains(r"confidential", case=False, na=False)
    ) &
    ~df_clean["salary_clean_text"].str.contains(r"RM|\$|MYR|SGD|฿", case=False, na=False)
)

# Set invalid salary records to missing in the extracted salary fields
df_clean.loc[mask_invalid_salary_text, ["salary_min", "salary_max", "salary_mid", "salary_monthly_est"]] = np.nan
df_clean.loc[mask_invalid_salary_text, "salary_has_number"] = 0

In [257]:
# Preview records treated as invalid salary text
df_clean.loc[
    mask_invalid_salary_text,
    ["job_title", "salary_clean_text", "salary_min", "salary_max", "salary_mid", "salary_monthly_est"]
].head(30)

,job_title,salary_clean_text,salary_min,salary_max,salary_mid,salary_monthly_est
1191,ETL Developer (Financial Institution),Attractive Remuneration Package,NaN,NaN,NaN,NaN
2447,Staff Engineer Maintenance Molding,bonuses,NaN,NaN,NaN,NaN
4524,Senior Engineer Maintenance Wire Bond,bonuses,NaN,NaN,NaN,NaN
5035,Staff Engineer Process Stability,bonuses,NaN,NaN,NaN,NaN
5131,Workforce Planner,Competitive Salary and Benefits,NaN,NaN,NaN,NaN
6055,Senior Staff Engineer Process Stability,bonuses,NaN,NaN,NaN,NaN
6725,Financial Controller/Plant Controller (Ipoh),Attractive remuneration package,NaN,NaN,NaN,NaN
7732,Senior Staff Engineer Industrial Engineering,bonuses,NaN,NaN,NaN,NaN
8013,Software Engineer - Fullstack,Competitive Remuneration & Benefits,NaN,NaN,NaN,NaN
8067,Senior Account Manager/ Account Manager [Growth],Attractive Salary Package,NaN,NaN,NaN,NaN


In [258]:
df_clean["salary_monthly_est"].describe()

count     31504.000000
mean       4703.524770
std        6845.518763
min           1.000000
25%        3000.000000
50%        3750.000000
75%        5000.000000
max      440000.000000
Name: salary_monthly_est, dtype: float64

In [259]:
# Create a flag for salary values that fall outside the reasonable monthly range
df_clean["salary_suspicious_flag"] = 0

mask_suspicious_salary = (
    df_clean["salary_monthly_est"].notna() &
    (
        (df_clean["salary_monthly_est"] < 300) |
        (df_clean["salary_monthly_est"] > 100000)
    )
)

df_clean.loc[mask_suspicious_salary, "salary_suspicious_flag"] = 1

df_clean["salary_suspicious_flag"].value_counts()

salary_suspicious_flag
0    68973
1       51
Name: count, dtype: int64

In [260]:
# Inspect salary records flagged as suspicious
df_clean.loc[
    df_clean["salary_suspicious_flag"] == 1,
    ["job_title", "salary_clean_text", "salary_period", "salary_min", "salary_max", "salary_mid", "salary_monthly_est"]
].sort_values("salary_monthly_est", ascending=False)

,job_title,salary_clean_text,salary_period,salary_min,salary_max,salary_mid,salary_monthly_est
57111,Part Time Aircond Technician,"RM 2,000 - RM 3,000 per hour",hourly,2000.0,3000.0,2500.00,440000.000000
55849,Part Time Plumber,"RM 2,000 - RM 3,000 per hour",hourly,2000.0,3000.0,2500.00,440000.000000
55558,Part Time Driver,"RM 2,000 - RM 3,000 per hour",hourly,2000.0,3000.0,2500.00,440000.000000
57017,Part Time Pet Groomer,"RM 2,000 - RM 3,000 per hour",hourly,2000.0,3000.0,2500.00,440000.000000
67019,Freelance Trainers Needed for Online Webinars,"RM 2,000 - RM 3,000 per hour",hourly,2000.0,3000.0,2500.00,440000.000000
57776,Intern - Mobility Team (Cyberjaya),"RM 1,500 - RM 1,501 per hour",hourly,1500.0,1501.0,1500.50,264088.000000
40635,Process Integration Engineer,"RM 200,000 - RM 300,000 per month",monthly,200000.0,300000.0,250000.00,250000.000000
31945,Internship - Finance,"RM 800 - RM 1,000 per hour",hourly,800.0,1000.0,900.00,158400.000000
9959,Corporate Banking Lending Specialist,"RM 130,000 - RM 150,000 per month",monthly,130000.0,150000.0,140000.00,140000.000000
18934,Devops Engineer,$100000.00 - $150000.00 per month,monthly,100000.0,150000.0,125000.00,125000.000000


In [261]:
# Create the final monthly salary variable for analysis
df_clean["salary_monthly_final"] = df_clean["salary_monthly_est"].copy()

# Identify salary values that are not suitable for analysis
mask_invalid_for_analysis = (
    df_clean["salary_monthly_est"].notna() &
    (
        (df_clean["salary_monthly_est"] < 300) |   # unrealistically low
        (df_clean["salary_monthly_est"] > 100000)  # unrealistically high / unsuitable
    )
)

# Set invalid salary values to missing in the final analysis column
df_clean.loc[mask_invalid_for_analysis, "salary_monthly_final"] = np.nan

In [262]:
# Review the summary statistics of the final monthly salary variable
df_clean["salary_monthly_final"].describe()

count    31453.000000
mean      4604.729155
std       3211.222698
min        300.000000
25%       3000.000000
50%       3750.000000
75%       5000.000000
max      85000.000000
Name: salary_monthly_final, dtype: float64

In [263]:
# Create a flag to indicate whether the final monthly salary is usable for analysis
df_clean["salary_usable_flag"] = df_clean["salary_monthly_final"].notna().astype(int)
df_clean["salary_usable_flag"].value_counts()

print("Usable salary rows:", df_clean["salary_usable_flag"].sum())
print("Unusable salary rows:", (df_clean["salary_usable_flag"] == 0).sum())
print("Usable salary percentage:", round(df_clean["salary_usable_flag"].mean() * 100, 2), "%")

Usable salary rows: 31453
Unusable salary rows: 37571
Usable salary percentage: 45.57 %


In [264]:
# Review the most frequent values in the cleaned location column
df_clean["location_clean"].value_counts().head(60)

location_clean
Kuala Lumpur                              15390
Petaling                                   9746
Johor Bahru District                       4480
Selangor                                   2842
Shah Alam/Subang                           2621
Kuala Lumpur City Centre                   2112
Penang                                     1930
Penang Island                              1885
Seberang Perai                             1796
Klang District                             1427
Hulu Langat                                 958
Kajang/Bangi/Serdang                        937
Sepang                                      865
Kota Kinabalu District                      805
Kinta District                              776
Kuching Division                            770
Johor                                       729
Seremban District                           638
Kulai District                              628
Petaling Jaya, Selangor                     519
Bangsar South            

In [265]:
# Clean location using state rule-based mapping
def map_location_to_state(loc):
    if pd.isna(loc):
        return np.nan
    
    loc = str(loc).strip().lower()
    
    # Federal Territories
    if any(x in loc for x in [
        "kuala lumpur", "kl eco city", "kuala lumpur city centre",
        "kuala lumpur sentral", "bukit jalil", "bangsar", "bangsar south",
        "cheras", "kepong", "mid valley city", "mont kiara",
        "bukit damansara", "bandar malaysia", "taman tun dr ismail",
        "setapak", "bukit bintang", "kuchai lama", "taman desa",
        "sungai besi", "sentul", "tun razak exchange", "brickfields",
        "segambut", "bandar tasik selatan", "maluri", "kampung malaysia raya",
        "sri hartamas"
    ]):
        return "Kuala Lumpur"
    
    if "putrajaya" in loc:
        return "Putrajaya"
    
    if "labuan" in loc:
        return "Labuan"
    
    # Selangor
    if any(x in loc for x in [
        "petaling", "petaling jaya", "shah alam", "subang", "subang jaya",
        "kajang", "bangi", "serdang", "sepang", "hulu langat",
        "klang", "port klang", "gombak", "puchong", "kuala langat",
        "kuala selangor", "selangor", "ampang"
    ]):
        return "Selangor"
    
    # Johor
    if any(x in loc for x in [
        "johor bahru", "johor", "kulai", "muar", "batu pahat", "senai",
        "pontian", "kluang", "kota tinggi", "segamat", "tangkak"
    ]):
        return "Johor"
    
    # Penang
    if any(x in loc for x in [
        "penang", "penang island", "seberang perai", "bayan lepas"
    ]):
        return "Penang"
    
    # Perak
    if any(x in loc for x in [
        "perak", "ipoh", "kinta", "manjung", "kampar",
        "larut, matang and selama district", "muallim"
    ]):
        return "Perak"
    
    # Sabah
    if any(x in loc for x in [
        "sabah", "kota kinabalu", "tawau", "sandakan",
        "penampang", "lahad datu"
    ]):
        return "Sabah"
    
    # Sarawak
    if any(x in loc for x in [
        "sarawak", "kuching", "miri", "bintulu", "sibu", "samarahan"
    ]):
        return "Sarawak"
    
    # Negeri Sembilan
    if any(x in loc for x in [
        "negeri sembilan", "seremban", "port dickson", "tampin"
    ]):
        return "Negeri Sembilan"
    
    # Melaka
    if any(x in loc for x in [
        "melaka", "melaka tengah", "alor gajah", "jasin"
    ]):
        return "Melaka"
    
    # Pahang
    if any(x in loc for x in [
        "pahang", "kuantan", "bentong", "temerloh"
    ]):
        return "Pahang"
    
    # Kedah
    if any(x in loc for x in [
        "kedah", "kulim", "kuala muda", "kota setar", "langkawi",
        "baling", "kubang pasu"
    ]):
        return "Kedah"
    
    # Kelantan
    if any(x in loc for x in [
        "kelantan", "kota bharu", "gua musang"
    ]):
        return "Kelantan"
    
    # Terengganu
    if any(x in loc for x in [
        "terengganu", "kuala terengganu", "kemaman", "besut"
    ]):
        return "Terengganu"
    
    # fallback
    if loc == "others":
        return "Others"
    if loc == "malaysia":
        return "Malaysia"
    
    return "Unknown"

In [266]:
df_clean["state_clean"] = df_clean["location_clean"].apply(map_location_to_state)

df_clean[["location_clean", "state_clean"]].sample(30, random_state=42)

,location_clean,state_clean
14886,Seberang Perai,Penang
21842,Kuala Lumpur,Kuala Lumpur
53575,Mont Kiara,Kuala Lumpur
50853,Kuala Lumpur,Kuala Lumpur
5662,Kuala Lumpur,Kuala Lumpur
57533,Selangor,Selangor
56381,Sepang,Selangor
36229,Penang Island,Penang
20851,Klang District,Selangor
836,Kuala Lumpur,Kuala Lumpur


In [267]:
df_clean["state_clean"].value_counts(dropna=False)

state_clean
Selangor           23103
Kuala Lumpur       21779
Johor               7466
Penang              6373
Sarawak             2019
Perak               1454
Sabah               1345
Kedah               1137
Melaka              1114
Negeri Sembilan     1002
Pahang               601
Unknown              536
Terengganu           258
Putrajaya            242
Kelantan             215
Others               198
Malaysia              98
Labuan                84
Name: count, dtype: int64

In [268]:
df_clean.loc[
    df_clean["state_clean"] == "Unknown",
    "location_clean"
].value_counts().head(50)

location_clean
Desa Parkcity             17
Limbang Division          17
Wangsa Maju               16
Sabak Bernam              15
Pekan District            15
Bandar Menjalara          15
Pudu                      14
Taman Connaught           14
Batang Padang District    14
Kerian District           12
Jerantut District         12
Kuala Kangsar District    11
Marang District           11
Shamelin Perkasa          10
Bandar Sri Permaisuri     10
Dungun                    10
Raub District             10
Sipitang District          9
Taman OUG                  9
Maran District             9
Rompin                     9
Kudat District             9
Kota Belud District        8
Keningau District          8
Mersing District           8
Tanah Merah                8
Bandar Baharu              7
Alam Damai                 7
Jempol District            7
Pantai Dalam               7
Mukah Division             7
Serian Division            7
Kuala Pilah District       7
Kangar                     7

In [269]:
def fill_remaining_unknown_state(loc, current_state):
    if pd.isna(loc):
        return current_state
    
    loc = str(loc).strip().lower()
    
    if current_state != "Unknown":
        return current_state
    
    # Kuala Lumpur
    if any(x in loc for x in [
        "desa parkcity", "wangsa maju", "bandar menjalara", "pudu",
        "taman connaught", "shamelin perkasa", "bandar sri permaisuri",
        "taman oug", "pantai dalam", "pusat bandar damansara",
        "happy garden", "seputeh", "kampung pandan", "taman midah"
    ]):
        return "Kuala Lumpur"
    
    # Selangor
    if "sabak bernam" in loc:
        return "Selangor"
    
    # Pahang
    if any(x in loc for x in [
        "pekan", "jerantut", "raub", "maran", "rompin",
        "lipis", "cameron highlands"
    ]):
        return "Pahang"
    
    # Perak
    if any(x in loc for x in [
        "batang padang", "kerian", "kuala kangsar"
    ]):
        return "Perak"
    
    # Terengganu
    if any(x in loc for x in [
        "marang", "dungun", "setiu"
    ]):
        return "Terengganu"
    
    # Sabah
    if any(x in loc for x in [
        "sipitang", "kudat", "kota belud", "keningau",
        "ranau", "beaufort", "putatan"
    ]):
        return "Sabah"
    
    # Johor
    if "mersing" in loc:
        return "Johor"
    
    # Kelantan
    if any(x in loc for x in [
        "tanah merah", "pasir puteh"
    ]):
        return "Kelantan"
    
    # Sarawak
    if any(x in loc for x in [
        "limbang", "mukah", "serian", "sri aman", "sarikei"
    ]):
        return "Sarawak"
    
    # Negeri Sembilan
    if any(x in loc for x in [
        "jempol", "kuala pilah", "rembau", "jelebu"
    ]):
        return "Negeri Sembilan"
    
    # Kedah
    if "bandar baharu" in loc:
        return "Kedah"
    
    # Perlis
    if "kangar" in loc:
        return "Perlis"
    
    return "Unknown"

In [270]:
df_clean["state_clean"] = df_clean.apply(
    lambda row: fill_remaining_unknown_state(row["location_clean"], row["state_clean"]),
    axis=1
)

In [271]:
df_clean["state_clean"].value_counts(dropna=False)

state_clean
Selangor           23118
Kuala Lumpur       21916
Johor               7474
Penang              6373
Sarawak             2061
Perak               1491
Sabah               1394
Kedah               1144
Melaka              1114
Negeri Sembilan     1028
Pahang               679
Terengganu           274
Putrajaya            242
Kelantan             229
Others               198
Unknown               99
Malaysia              98
Labuan                84
Perlis                 8
Name: count, dtype: int64

In [272]:
df_clean.loc[
    df_clean["state_clean"] == "Unknown",
    "location_clean"
].value_counts().head(50)

location_clean
Alam Damai                     7
Pasir Mas District             4
Taman Danau Desa               4
Pandan Perdana                 4
Kampung Baru                   4
Kota Marudu District           4
Pendang District               4
Bachok                         4
Sik District                   4
Padang Besar                   3
Perlis                         3
Tuaran District                3
Kapit Division                 3
Machang                        3
Bagan Datuk District           3
Taman United                   3
Salak South                    3
Setiawangsa                    3
Titiwangsa                     2
Chow Kit                       2
Kampung Sungai Penchala        2
Bukit Kiara                    2
Semporna District              2
Papar District                 2
Taman Wahyu                    2
Tumpat                         2
Kunak District                 2
Kuala Nerus District           2
Wangsa Melawati                1
Medan Tuanku                

In [273]:
def fill_last_unknown_state(loc, current_state):
    if pd.isna(loc):
        return current_state
    
    loc = str(loc).strip().lower()
    
    if current_state != "Unknown":
        return current_state
    
    # Kuala Lumpur
    if any(x in loc for x in [
        "alam damai", "taman danau desa", "pandan perdana", "kampung baru",
        "taman united", "salak south", "setiawangsa", "titiwangsa",
        "chow kit", "kampung sungai penchala", "bukit kiara",
        "taman wahyu", "wangsa melawati", "medan tuanku",
        "jalan cochrane", "taman len seng", "bukit tunku"
    ]):
        return "Kuala Lumpur"
    
    # Selangor
    if any(x in loc for x in [
        "lembah keramat", "taman sri sinar"
    ]):
        return "Selangor"
    
    # Sabah
    if any(x in loc for x in [
        "kota marudu", "tuaran", "semporna", "papar", "kunak", "beluran"
    ]):
        return "Sabah"
    
    # Kedah
    if any(x in loc for x in [
        "pendang", "sik", "yan"
    ]):
        return "Kedah"
    
    # Perlis
    if any(x in loc for x in [
        "padang besar", "perlis"
    ]):
        return "Perlis"
    
    # Sarawak
    if any(x in loc for x in [
        "kapit", "betong"
    ]):
        return "Sarawak"
    
    # Kelantan
    if any(x in loc for x in [
        "pasir mas", "bachok", "machang", "tumpat", "jeli"
    ]):
        return "Kelantan"
    
    # Perak
    if "bagan datuk" in loc:
        return "Perak"
    
    # Terengganu
    if "kuala nerus" in loc:
        return "Terengganu"
    
    # Pahang
    if "bera" in loc:
        return "Pahang"
    
    # Penang
    if "tanjung bungah" in loc or "teluk bahang" in loc:
        return "Penang"
    
    return "Unknown"

In [274]:
# Map location values into state level categories
def map_location_to_state(loc):
    if pd.isna(loc):
        return np.nan
    
    loc = str(loc).strip().lower()
    
    # Federal Territories
    if any(x in loc for x in [
        "kuala lumpur", "kl eco city", "kuala lumpur city centre",
        "kuala lumpur sentral", "bukit jalil", "bangsar", "bangsar south",
        "cheras", "kepong", "mid valley city", "mont kiara",
        "bukit damansara", "bandar malaysia", "taman tun dr ismail",
        "setapak", "bukit bintang", "kuchai lama", "taman desa",
        "sungai besi", "sentul", "tun razak exchange", "brickfields",
        "segambut", "bandar tasik selatan", "maluri", "kampung malaysia raya",
        "sri hartamas", "desa parkcity", "wangsa maju", "bandar menjalara",
        "pudu", "taman connaught", "shamelin perkasa", "bandar sri permaisuri",
        "taman oug", "pantai dalam", "pusat bandar damansara",
        "happy garden", "seputeh", "kampung pandan", "taman midah",
        "alam damai", "taman danau desa", "pandan perdana", "kampung baru",
        "taman united", "salak south", "setiawangsa", "titiwangsa",
        "chow kit", "kampung sungai penchala", "bukit kiara",
        "taman wahyu", "wangsa melawati", "medan tuanku",
        "jalan cochrane", "taman len seng", "bukit tunku"
    ]):
        return "Kuala Lumpur"
    
    if "putrajaya" in loc:
        return "Putrajaya"
    
    if "labuan" in loc:
        return "Labuan"
    
    # Selangor
    if any(x in loc for x in [
        "petaling", "petaling jaya", "shah alam", "subang", "subang jaya",
        "kajang", "bangi", "serdang", "sepang", "hulu langat",
        "klang", "port klang", "gombak", "puchong", "kuala langat",
        "kuala selangor", "selangor", "ampang", "sabak bernam",
        "lembah keramat", "taman sri sinar"
    ]):
        return "Selangor"
    
    # Johor
    if any(x in loc for x in [
        "johor bahru", "johor", "kulai", "muar", "batu pahat", "senai",
        "pontian", "kluang", "kota tinggi", "segamat", "tangkak", "mersing"
    ]):
        return "Johor"
    
    # Penang
    if any(x in loc for x in [
        "penang", "penang island", "seberang perai", "bayan lepas",
        "tanjung bungah", "teluk bahang"
    ]):
        return "Penang"
    
    # Perak
    if any(x in loc for x in [
        "perak", "ipoh", "kinta", "manjung", "kampar",
        "larut, matang and selama district", "muallim",
        "batang padang", "kerian", "kuala kangsar", "bagan datuk"
    ]):
        return "Perak"
    
    # Sabah
    if any(x in loc for x in [
        "sabah", "kota kinabalu", "tawau", "sandakan",
        "penampang", "lahad datu", "sipitang", "kudat",
        "kota belud", "keningau", "ranau", "beaufort",
        "putatan", "kota marudu", "tuaran", "semporna",
        "papar", "kunak", "beluran"
    ]):
        return "Sabah"
    
    # Sarawak
    if any(x in loc for x in [
        "sarawak", "kuching", "miri", "bintulu", "sibu", "samarahan",
        "limbang", "mukah", "serian", "sri aman", "sarikei",
        "kapit", "betong"
    ]):
        return "Sarawak"
    
    # Negeri Sembilan
    if any(x in loc for x in [
        "negeri sembilan", "seremban", "port dickson", "tampin",
        "jempol", "kuala pilah", "rembau", "jelebu"
    ]):
        return "Negeri Sembilan"
    
    # Melaka
    if any(x in loc for x in [
        "melaka", "melaka tengah", "alor gajah", "jasin"
    ]):
        return "Melaka"
    
    # Pahang
    if any(x in loc for x in [
        "pahang", "kuantan", "bentong", "temerloh", "pekan",
        "jerantut", "raub", "maran", "rompin", "lipis",
        "cameron highlands", "bera"
    ]):
        return "Pahang"
    
    # Kedah
    if any(x in loc for x in [
        "kedah", "kulim", "kuala muda", "kota setar", "langkawi",
        "baling", "kubang pasu", "bandar baharu", "pendang",
        "sik", "yan"
    ]):
        return "Kedah"
    
    # Kelantan
    if any(x in loc for x in [
        "kelantan", "kota bharu", "gua musang", "tanah merah",
        "pasir puteh", "pasir mas", "bachok", "machang",
        "tumpat", "jeli"
    ]):
        return "Kelantan"
    
    # Terengganu
    if any(x in loc for x in [
        "terengganu", "kuala terengganu", "kemaman", "besut",
        "marang", "dungun", "setiu", "kuala nerus"
    ]):
        return "Terengganu"
    
    # Perlis
    if any(x in loc for x in [
        "kangar", "padang besar", "perlis"
    ]):
        return "Perlis"
    
    # Fallback
    if loc == "others":
        return "Others"
    if loc == "malaysia":
        return "Malaysia"
    
    return "Unknown"

In [275]:
# Apply rule based location mapping to create a cleaned state column
df_clean["state_clean"] = df_clean["location_clean"].apply(map_location_to_state)

# Review the distribution of mapped state values
df_clean["state_clean"].value_counts(dropna=False)

state_clean
Selangor           23120
Kuala Lumpur       21959
Johor               7474
Penang              6374
Sarawak             2065
Perak               1500
Sabah               1408
Kedah               1154
Melaka              1114
Negeri Sembilan     1028
Pahang               681
Terengganu           274
Putrajaya            242
Kelantan             237
Others               198
Malaysia              98
Labuan                84
Perlis                14
Name: count, dtype: int64

In [276]:
df_clean["state_clean"] = df_clean.apply(
    lambda row: fill_last_unknown_state(row["location_clean"], row["state_clean"]),
    axis=1
)

In [277]:
df_clean["state_clean"].value_counts(dropna=False)

state_clean
Selangor           23120
Kuala Lumpur       21959
Johor               7474
Penang              6374
Sarawak             2065
Perak               1500
Sabah               1408
Kedah               1154
Melaka              1114
Negeri Sembilan     1028
Pahang               681
Terengganu           274
Putrajaya            242
Kelantan             237
Others               198
Malaysia              98
Labuan                84
Perlis                14
Name: count, dtype: int64

In [278]:
df_clean.loc[
    df_clean["state_clean"] == "Unknown",
    "location_clean"
].value_counts().head(50)

Series([], Name: count, dtype: int64)

In [279]:
# Create a cleaned role column and replace missing values with 'Unknown'
df_clean["role_clean"] = df_clean["role"].copy()

df_clean["role_clean"] = df_clean["role_clean"].fillna("Unknown")
df_clean["role_clean"] = df_clean["role_clean"].str.strip()
df_clean["role_clean"] = df_clean["role_clean"].str.replace(r"\s+", " ", regex=True)

df_clean["role_clean"].value_counts().head(30)

role_clean
Unknown                                         2252
account-executive                               1483
sales-executive                                 1221
accounts-executive                              1198
administration-officer                          1116
Executive                                        951
human-resource-executive                         919
administrative-executive                         826
finance-executive                                736
marketing-executive                              643
administration                                   625
engineer                                         556
project-engineer                                 545
account-assistant                                509
software-engineer                                499
accountant                                       447
customer-service-executive                       446
assistant-manager                                436
manager                            

In [280]:
print("Unique roles before cleaning:", df_clean["role"].nunique(dropna=True))
print("Unique roles after basic cleaning:", df_clean["role_clean"].nunique(dropna=True))

Unique roles before cleaning: 5484
Unique roles after basic cleaning: 5485


In [281]:
df_clean["role_clean"].value_counts().head(30)

role_clean
Unknown                                         2252
account-executive                               1483
sales-executive                                 1221
accounts-executive                              1198
administration-officer                          1116
Executive                                        951
human-resource-executive                         919
administrative-executive                         826
finance-executive                                736
marketing-executive                              643
administration                                   625
engineer                                         556
project-engineer                                 545
account-assistant                                509
software-engineer                                499
accountant                                       447
customer-service-executive                       446
assistant-manager                                436
manager                            

In [282]:
# Standardize role formatting
df_clean["role_clean"] = df_clean["role_clean"].str.lower()
df_clean["role_clean"] = df_clean["role_clean"].str.strip()
df_clean["role_clean"] = df_clean["role_clean"].str.replace(r"\s+", " ", regex=True)
df_clean["role_clean"] = df_clean["role_clean"].str.replace("_", "-", regex=False)
df_clean["role_clean"] = df_clean["role_clean"].str.replace(r"-{2,}", "-", regex=True)
df_clean["role_clean"] = df_clean["role_clean"].str.replace(r"-\d+$", "", regex=True)
df_clean["role_clean"] = df_clean["role_clean"].str.strip("-").str.strip()

df_clean["role_clean"].value_counts().head(50)

role_clean
unknown                                         2252
account-executive                               1483
sales-executive                                 1221
accounts-executive                              1198
administration-officer                          1116
executive                                        951
human-resource-executive                         919
administrative-executive                         826
finance-executive                                736
marketing-executive                              643
administration                                   625
engineer                                         556
project-engineer                                 545
account-assistant                                509
software-engineer                                499
accountant                                       447
customer-service-executive                       446
assistant-manager                                436
manager                            

In [283]:
# Create a cleaned copy of the job description column
df_clean["descriptions_clean"] = df_clean["descriptions"].copy()

# Normalize whitespace in the description text
df_clean["descriptions_clean"] = df_clean["descriptions_clean"].str.replace(r"\s+", " ", regex=True)
df_clean["descriptions_clean"] = df_clean["descriptions_clean"].str.strip()

In [284]:
# Display original and cleaned description text for comparison
pd.set_option("display.max_colwidth", None)
df_clean.loc[10:14, ["descriptions", "descriptions_clean"]]

,descriptions,descriptions_clean
10,"Educational Centre\nEstablished since 2019, total around 20 staffs\nOffice near to KL Gateway Mall, reachable by public transport\nResponsibilities:\nAssist overseas students application for local and private universities or colleges in Malaysia\nAnswer student enquiries\nProvide advice and guidance\nRequirements\nFresh Graduates are welcome to apply\n1-2 years of working experience are welcome to apply\nTraining will be provided\nSalary: RM3,000 - RM3,200 + Commissions\nWorking Hours:\n9am - 6pm (Mon-Fri)","Educational Centre Established since 2019, total around 20 staffs Office near to KL Gateway Mall, reachable by public transport Responsibilities: Assist overseas students application for local and private universities or colleges in Malaysia Answer student enquiries Provide advice and guidance Requirements Fresh Graduates are welcome to apply 1-2 years of working experience are welcome to apply Training will be provided Salary: RM3,000 - RM3,200 + Commissions Working Hours: 9am - 6pm (Mon-Fri)"
11,"​Job Responsibilities\nFinancial Pre-Qualification\nPerform accounting ratio analysis based on the latest audited financial statement or management account obtained.\nLiaise with relevant departments in resolving any incomplete date to meet the objective.\nGenerate the Financial Pre-qualification for Finance Manager review and approval before circulating to the relevant department.\nCar Park Income\nReconcile parking collections against bank statements to ensure completeness in collections.\nLiaise with relevant departments to resolve all reconciling items and follow-up in obtaining the relevant supporting documents.\nPrepare monthly parking collections reports for management review and approval as well as filing for documentation completeness.\nTo handle both external/Internal audit on car park income matters.\nOther matters\nAssist Finance Manager to perform:\na) Monthly variance analysis on revenue and operating expenses\nb) Consolidate monthly results for country reporting\nAssist relevant department to request for Asset Number when raising PO.\nPerform any other duties, which may be assigned or directed by the line of report from time to time.\nSupport other entities within the Group or special project as and when assigned.\nRequirements\nAt least a Bachelor's Degree In Finance, Accountancy or equivalent.\nCompleted ACCA or MICPA. Finalists of ACCA or MICPA with the relevant work experience are encouraged to apply.\nMinimum 2 years of relevant work experience.\nProficiency in Microsoft Office such as Excel and PowerPoint.\nGood analytical, communication and interpersonal skills.\nCommitted and a strong team player with sense of urgency.\nAble to work independently in a dynamic environment.\nFresh graduates are encouraged to buy.\nLocation : Kuala Lumpur - Naza Tower, Persiaran KLCC","​Job Responsibilities Financial Pre-Qualification Perform accounting ratio analysis based on the latest audited financial statement or management account obtained. Liaise with relevant departments in resolving any incomplete date to meet the objective. Generate the Financial Pre-qualification for Finance Manager review and approval before circulating to the relevant department. Car Park Income Reconcile parking collections against bank statements to ensure completeness in collections. Liaise with relevant departments to resolve all reconciling items and follow-up in obtaining the relevant supporting documents. Prepare monthly parking collections reports for management review and approval as well as filing for documentation completeness. To handle both external/Internal audit on car park income matters. Other matters Assist Finance Manager to perform: a) Monthly variance analysis on revenue and operating expenses b) Consolidate monthly results for country reporting Assist relevant department to request for Asset Number when raising PO. Perform any other duties, which may be assigned or directed by the line of

In [285]:
# Description length feature
df_clean["description_length"] = df_clean["descriptions_clean"].str.len()

df_clean["description_length"].describe()

count    69024.000000
mean      2189.331928
std       1534.035794
min          2.000000
25%       1164.000000
50%       1764.000000
75%       2766.000000
max      16256.000000
Name: description_length, dtype: float64

In [286]:
# Define patterns that indicate invalid or non informative job descriptions
invalid_description_patterns = [
    r"^#name\?$",
    r"^back apply$",
    r"^ok$",
    r"^\d+(\s+\d+)*$"
]

pattern_combined = "|".join(invalid_description_patterns)

# Create a flag for descriptions that match invalid text patterns
df_clean["description_invalid_flag"] = df_clean["descriptions_clean"].str.lower().str.match(
    pattern_combined,
    na=False
).astype(int)

df_clean["description_invalid_flag"].value_counts()

description_invalid_flag
0    69003
1       21
Name: count, dtype: int64

In [287]:
# Inspect records with the shortest cleaned job descriptions
df_clean.loc[
    df_clean["description_invalid_flag"] == 1,
    ["job_title", "descriptions_clean", "description_length"]
].head(30)

,job_title,descriptions_clean,description_length
1741,Accountant cum HR Executive,#NAME?,6
2307,accounting,1 2 3,5
3233,Runner,#NAME?,6
4236,Administrative Assistant,#NAME?,6
4724,Bank Reconciliation Specialist,#NAME?,6
5914,Quantity Surveyor / Contracts Executive,#NAME?,6
8580,Account cum HR Assistant,#NAME?,6
10235,Billing Executive,#NAME?,6
10544,PURCHASING EXECUTIVE,#NAME?,6
10788,Finance & Account Executive,#NAME?,6


In [288]:
# Set invalid job descriptions to missing
df_clean.loc[df_clean["description_invalid_flag"] == 1, "descriptions_clean"] = np.nan
df_clean.loc[df_clean["description_invalid_flag"] == 1, "description_length"] = np.nan

In [289]:
# Preview records where invalid descriptions were removed
df_clean.loc[
    df_clean["description_invalid_flag"] == 1,
    ["job_title", "descriptions_clean", "description_length"]
].head(30)

,job_title,descriptions_clean,description_length
1741,Accountant cum HR Executive,NaN,NaN
2307,accounting,NaN,NaN
3233,Runner,NaN,NaN
4236,Administrative Assistant,NaN,NaN
4724,Bank Reconciliation Specialist,NaN,NaN
5914,Quantity Surveyor / Contracts Executive,NaN,NaN
8580,Account cum HR Assistant,NaN,NaN
10235,Billing Executive,NaN,NaN
10544,PURCHASING EXECUTIVE,NaN,NaN
10788,Finance & Account Executive,NaN,NaN


In [290]:
# Inspect the shortest remaining valid job descriptions
df_clean.loc[
    df_clean["descriptions_clean"].notna(),
    ["job_title", "descriptions_clean", "description_length"]
].sort_values("description_length", ascending=True).head(30)

,job_title,descriptions_clean,description_length
52478,Internship,Retail company,14.0
15973,IT / MULTIMEDIA,ADMIN ASSISTANT,15.0
15924,Intern HR & Administration,"Cyberjaya, Malaysia",19.0
53827,Accounting assistant,Accounting assistant,20.0
23235,SALES EXECUTIVE,INTERNATIONAL OFFICER,21.0
8027,Admin Vacancy,Hardware Shop Admin Role,24.0
15749,INTERNATIONAL OFFICER,SAFETY AND HEALTH OFFICER,25.0
18302,Account executive,Finding account executive,25.0
15929,HUMAN RESOURCE EXECUTIVE,CUSTOMER SERVICE EXECUTIVE,26.0
38459,Driver,Lorry driver with GDL Permit,28.0


In [291]:
# Export the final cleaned dataset to a CSV file
df = pd.read_csv("../data/jobstreet_cleaned_final.csv")
print("Saved as jobstreet_cleaned_final.csv")

Saved as jobstreet_cleaned_final.csv
